# Stage-2 — L3/L4 매핑 (rev.2, 감쇠 EM + 시드 보강)

**rev.1 민감도 결과 반영**: 무작위 초기화 런들이 서로 0.3~3.3%만 일치 → EM이 데이터로 식별되지 않음(초기값이 결과를 결정). 원인은 M-step에서 centroid가 L3 시드로부터 이탈(코사인 0.605까지 하락).

**개정 (A+C)**
- **A. 감쇠 EM** — centroid를 `normalize(α·시드 + (1−α)·군집평균)`로 갱신. α=0.4 기본. 초기화는 항상 시드 고정(무작위 초기화는 부적격 명세로 제외).
- **C. L3 시드 보강** — 시드 텍스트에 해당 L3의 대표 L4 예시를 결합해 앵커 강화. HLD 노드는 보강 제외(비워지는 것이 목표).

**민감도 3축 × 2회** (2회 = 90% 부트스트랩 부분표본 2종 — 감쇠 EM은 결정론적이므로 진짜 반복은 표본 변동으로 구성)
1. 카드 텍스트: bilingual / english
2. L3 시드: enriched / plain
3. 감쇠 계수 α: 0.4 / 0.6

| 셀 | 담당 | 내용 |
|---|---|---|
| U0–U2 | 🔵 | 설정 · 로드 · 임베딩(카드 2종, 시드 2종) |
| U3 | 🔵 | 감쇠 EM 엔진 + α 진단 |
| U4 | 🔵 | **1단계 자동 배정** + 민감도 3축×2회 + 안정성 |
| **GATE-S2** | 🟢 | hold 심의 → `hold_decisions.json` |
| U5 | 🔵 | **2단계 강제 배정** |
| U6 | 🔵 | 2단계 민감도 3축×2회 |
| U7 | 🔵 | 검증 + 산출물 + HTML |

In [1]:
# U0 — 설정
import json, os, re, hashlib, platform, datetime, csv
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, scipy, pandas as pd

ROOT=Path.cwd()
while not (ROOT/'data/experiments/stage1/out/master.json').exists():
    if ROOT.parent==ROOT: raise FileNotFoundError('Stage-1 산출물 없음')
    ROOT=ROOT.parent
os.chdir(ROOT); print('repo root:',ROOT)
S1=ROOT/'data/experiments/stage1/out'
S2=ROOT/'data/experiments/stage2'; DEC=S2/'decisions'; OUT=S2/'out'
for p in (S2,DEC,OUT): p.mkdir(parents=True,exist_ok=True)
HIER=ROOT/'public/data/releases/v2.18.0-rc/hierarchy.json'
_m=ROOT/'tmp/models/bge-m3'
MODEL=str(_m) if (_m/'modules.json').exists() else 'BAAI/bge-m3'
if (_m/'modules.json').exists(): os.environ['HF_HUB_OFFLINE']='1'

CFG=dict(model=MODEL, max_seq_length=256, batch_size=32,
    alpha=0.4, alpha_alt=0.6, em_iters=30, em_tol=1e-5,
    margin_hold=0.03, boot_frac=0.9, boot_seeds=[20260807,20260808],
    seed_exemplars=8, sets=['master','c80','c70'])
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()[:16]
def unit(x): return x/np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-12)
print(json.dumps({k:v for k,v in CFG.items() if k!='model'},ensure_ascii=False))

repo root: /Users/deep1003/data3/RAI-Risk-Taxonomy
{"max_seq_length": 256, "batch_size": 32, "alpha": 0.4, "alpha_alt": 0.6, "em_iters": 30, "em_tol": 1e-05, "margin_hold": 0.03, "boot_frac": 0.9, "boot_seeds": [20260807, 20260808], "seed_exemplars": 8, "sets": ["master", "c80", "c70"]}


In [2]:
# U1 — Stage-1 세트 + L3 로드
SETS={s:json.load(open(S1/(s+'.json')))['cards'] for s in CFG['sets']}
for s in CFG['sets']: print(s.upper(), len(SETS[s]))
H=json.load(open(HIER))
L3=[n for n in H['nodes'] if str(n['node_id']).startswith('RAI3')]
assert len(L3)==56
HOLD_L3={n['node_id'] for n in L3 if 'HLD' in n['node_id']}
l3idx={n['node_id']:i for i,n in enumerate(L3)}
OLDMAP={c['l4_id']:c.get('primary_l3_id')
        for c in json.load(open(ROOT/'public/data/releases/v2.18.0-rc/cards.json'))['cards']}
print('L3',len(L3),'| HLD',sorted(HOLD_L3))

MASTER 1612
C80 1369
C70 937
L3 56 | HLD ['RAI3-A-HLD-01', 'RAI3-G-HLD-01']


In [3]:
# U2 — 임베딩: 카드 2종(bilingual/english) + L3 시드 2종(plain/enriched)
def card_text(c,v):
    if v=='english': return (c.get('label_en','') or '')+'. '+(c.get('definition_en','') or '')
    return ((c.get('label_en','') or '')+'. '+(c.get('definition_en','') or '')+' / '
            +(c.get('label_ko','') or '')+'. '+(c.get('definition_ko','') or ''))
_enc=[None]
def enc():
    if _enc[0] is None:
        from sentence_transformers import SentenceTransformer
        m=SentenceTransformer(CFG['model']); m.max_seq_length=CFG['max_seq_length']; _enc[0]=m
    return _enc[0]
def embed(texts,tag):
    th=hashlib.sha1((tag+chr(30)+chr(30).join(texts)).encode()).hexdigest()[:12]
    p=OUT/('emb_'+tag+'_'+th+'.npy')
    if p.exists(): return np.load(p)
    v=enc().encode(texts,normalize_embeddings=True,batch_size=CFG['batch_size'],show_progress_bar=True).astype('float32')
    np.save(p,v); return v

CARD={}; IDS={}
for s in CFG['sets']:
    IDS[s]=[c['l4_id'] for c in SETS[s]]
    for v in ('bilingual','english'):
        CARD[(s,v)]=embed([card_text(c,v) for c in SETS[s]], s+'_'+v)
def l3_plain(n):
    return ((n.get('label_en','') or '')+'. '+(n.get('definition_en','') or '')+' / '
            +(n.get('label_ko','') or '')+'. '+(n.get('definition_ko','') or ''))
SEED_PLAIN=embed([l3_plain(n) for n in L3],'l3_plain')

# --- C. 시드 보강: 기존 배정 카드 중 시드에 가장 가까운 k장의 라벨을 결합 (HLD 제외) ---
mb=SETS['master']; Xm=CARD[('master','bilingual')]
prev=defaultdict(list)
for i,c in enumerate(mb):
    p=OLDMAP.get(c['l4_id'])
    if p and p not in HOLD_L3: prev[p].append(i)
ex_texts=[]; ex_cnt=[]
for j,n in enumerate(L3):
    base=l3_plain(n)
    cand=prev.get(n['node_id'],[])
    if n['node_id'] in HOLD_L3 or not cand:
        ex_texts.append(base); ex_cnt.append(0); continue
    sims=Xm[cand]@SEED_PLAIN[j]
    top=[cand[t] for t in np.argsort(-sims)[:CFG['seed_exemplars']]]
    ex=' ; '.join((mb[t].get('label_en') or '')+' / '+(mb[t].get('label_ko') or '') for t in top)
    ex_texts.append(base+' Examples: '+ex); ex_cnt.append(len(top))
SEED_ENR=embed(ex_texts,'l3_enriched')
print('시드 보강: 예시 결합 L3', sum(1 for x in ex_cnt if x), '/', len(L3),
      '| 평균 예시수', round(float(np.mean([x for x in ex_cnt if x])),1))
print('plain vs enriched 시드 코사인 평균', round(float((SEED_PLAIN*SEED_ENR).sum(1).mean()),3))
SEEDS={'plain':SEED_PLAIN,'enriched':SEED_ENR}

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

시드 보강: 예시 결합 L3 54 / 56 | 평균 예시수 6.3
plain vs enriched 시드 코사인 평균 0.952


In [4]:
# U3 — 감쇠 EM 엔진 + α 진단
HOLD_MASK=np.array([n['node_id'] in HOLD_L3 for n in L3])   # HLD 노드 배정 후보 제외
NEG=np.where(HOLD_MASK,-1e9,0.0)
def dem(X, SD, alpha, iters=None, tol=None, fixed=None, fixedA=None):
    iters=iters or CFG['em_iters']; tol=tol or CFG['em_tol']
    MU=SD.copy(); K=len(SD); A=None
    for it in range(1,iters+1):
        S=X@MU.T+NEG; nA=S.argmax(1)
        if fixed is not None: nA[fixed]=fixedA[fixed]
        NEW=MU.copy()
        for k in range(K):
            m=X[nA==k]
            NEW[k]=unit(alpha*SD[k]+(1-alpha)*unit(m.mean(0))) if len(m) else SD[k]
        stop=np.allclose(NEW,MU,atol=tol) and (A is not None and (nA==A).all())
        MU=NEW; A=nA
        if stop: break
    S=X@MU.T+NEG; A=S.argmax(1)
    if fixed is not None: A[fixed]=fixedA[fixed]
    srt=np.sort(S,1)
    return dict(assign=A, margin=srt[:,-1]-srt[:,-2], sim=S, top3=np.argsort(-S,1)[:,:3],
                iters=it, drift=(MU*SD).sum(1))

X=CARD[('master','bilingual')]
A_seed=(X@SEED_ENR.T+NEG).argmax(1)
print('α 진단 (master, enriched seed)')
for a in (0.0,0.2,0.4,0.6,0.8):
    r=dem(X,SEED_ENR,a)
    print(f'  α={a}: iters {r["iters"]:>2} | 시드일치 {(r["assign"]==A_seed).mean()*100:5.1f}%'
          f' | drift_min {r["drift"].min():.3f} | med margin {np.median(r["margin"]):.4f}'
          f' | usedL3 {len(set(r["assign"])):2} | max군집 {Counter(r["assign"].tolist()).most_common(1)[0][1]}')

α 진단 (master, enriched seed)
  α=0.0: iters 23 | 시드일치  56.6% | drift_min 0.605 | med margin 0.0386 | usedL3 55 | max군집 84
  α=0.2: iters 14 | 시드일치  70.2% | drift_min 0.732 | med margin 0.0337 | usedL3 55 | max군집 98
  α=0.4: iters  9 | 시드일치  78.5% | drift_min 0.848 | med margin 0.0296 | usedL3 55 | max군집 115
  α=0.6: iters  7 | 시드일치  87.5% | drift_min 0.935 | med margin 0.0271 | usedL3 55 | max군집 142
  α=0.8: iters  5 | 시드일치  94.2% | drift_min 0.985 | med margin 0.0237 | usedL3 55 | max군집 167


In [5]:
# U4 — 1단계 자동 배정 + 민감도 3축 × 2회(90% 부트스트랩)
BASE=dict(card='bilingual', seed='enriched', alpha=CFG['alpha'])
AXES=[('base',      BASE['card'],  BASE['seed'],    BASE['alpha']),
      ('axis1_english',  'english', BASE['seed'],    BASE['alpha']),
      ('axis2_plainseed', BASE['card'],'plain',      BASE['alpha']),
      ('axis3_alpha06',  BASE['card'], BASE['seed'], CFG['alpha_alt'])]
rows=[]; RES1={}
for s in CFG['sets']:
    for ax,cv,sv,al in AXES:
        Xf=CARD[(s,cv)]; SD=SEEDS[sv]
        full=dem(Xf,SD,al); RES1[(s,ax,'full')]=full
        mg=full['margin']
        rows.append(dict(set=s,axis=ax,run='full',n=len(Xf),em_iters=full['iters'],
            median_margin=round(float(np.median(mg)),4),
            hold=int((mg<CFG['margin_hold']).sum()),
            hold_pct=round(float((mg<CFG['margin_hold']).mean()*100),1),
            used_l3=len(set(full['assign'])), empty_l3=len(L3)-len(set(full['assign'])),
            max_family=Counter(full['assign'].tolist()).most_common(1)[0][1],
            drift_min=round(float(full['drift'].min()),3)))
        for bs in CFG['boot_seeds']:                      # 2회 반복
            rng=np.random.default_rng(bs)
            sel=np.sort(rng.choice(len(Xf),int(len(Xf)*CFG['boot_frac']),replace=False))
            rb=dem(Xf[sel],SD,al); RES1[(s,ax,bs)]=(sel,rb)
            agree=float((rb['assign']==full['assign'][sel]).mean()*100)
            rows.append(dict(set=s,axis=ax,run=f'boot{bs}',n=len(sel),em_iters=rb['iters'],
                median_margin=round(float(np.median(rb['margin'])),4),
                hold=int((rb['margin']<CFG['margin_hold']).sum()),
                hold_pct=round(float((rb['margin']<CFG['margin_hold']).mean()*100),1),
                used_l3=len(set(rb['assign'])), empty_l3=len(L3)-len(set(rb['assign'])),
                max_family=Counter(rb['assign'].tolist()).most_common(1)[0][1],
                drift_min=round(float(rb['drift'].min()),3),
                boot_agreement_pct=round(agree,1)))
df1=pd.DataFrame(rows); df1.to_csv(OUT/'stage1_sensitivity.csv',index=False)
print(df1.to_string(index=False))

   set            axis          run    n  em_iters  median_margin  hold  hold_pct  used_l3  empty_l3  max_family  drift_min  boot_agreement_pct
master            base         full 1612         9         0.0296   814      50.5       55         1         115      0.848                 NaN
master            base boot20260807 1450         9         0.0303   719      49.6       55         1         103      0.848                94.8
master            base boot20260808 1450         8         0.0308   702      48.4       55         1         105      0.848                96.1
master   axis1_english         full 1612        10         0.0299   809      50.2       55         1         128      0.845                 NaN
master   axis1_english boot20260807 1450         9         0.0302   721      49.7       55         1         124      0.845                94.8
master   axis1_english boot20260808 1450        14         0.0312   707      48.8       55         1         111      0.845             

In [6]:
# U4b — 축 간 안정성 (base 대비 일치율·ARI)
try: from sklearn.metrics import adjusted_rand_score as ari
except ImportError:
    def ari(a,b):
        n=len(a); c=Counter(zip(a,b)); sa=Counter(a); sb=Counter(b)
        cmb=lambda x:x*(x-1)/2
        s=sum(cmb(v) for v in c.values()); A=sum(cmb(v) for v in sa.values()); B=sum(cmb(v) for v in sb.values())
        e=A*B/cmb(n); m=(A+B)/2
        return (s-e)/(m-e) if m!=e else 1.0
st=[]
for s in CFG['sets']:
    b=RES1[(s,'base','full')]['assign']
    for ax,_,_,_ in AXES:
        if ax=='base': continue
        a=RES1[(s,ax,'full')]['assign']
        st.append(dict(set=s,vs=ax,agreement_pct=round(float((a==b).mean()*100),1),
                       ARI=round(float(ari(b.tolist(),a.tolist())),3)))
df1s=pd.DataFrame(st); df1s.to_csv(OUT/'stage1_stability.csv',index=False)
print(df1s.to_string(index=False))
print('\n※ 부트스트랩 재현성은 위 표의 boot_agreement_pct 참조 (동일 조건 90% 부분표본)')

   set              vs  agreement_pct   ARI
master   axis1_english           75.2 0.570
master axis2_plainseed           69.9 0.495
master   axis3_alpha06           89.7 0.790
   c80   axis1_english           73.0 0.552
   c80 axis2_plainseed           68.2 0.483
   c80   axis3_alpha06           91.2 0.798
   c70   axis1_english           74.8 0.597
   c70 axis2_plainseed           66.0 0.444
   c70   axis3_alpha06           93.4 0.848

※ 부트스트랩 재현성은 위 표의 boot_agreement_pct 참조 (동일 조건 90% 부분표본)


In [7]:
# U4c — 1단계 확정 + hold 목록 (🟢 Claude 입력 생성)
STAGE1={}
for s in CFG['sets']:
    r=RES1[(s,'base','full')]; A=r['assign']; mg=r['margin']; t3=r['top3']
    out=[]
    for i,c in enumerate(SETS[s]):
        hold=bool(mg[i]<CFG['margin_hold'])
        out.append(dict(l4_id=c['l4_id'], l3_stage1=None if hold else L3[A[i]]['node_id'],
            margin=round(float(mg[i]),4), hold=hold, prev_l3=OLDMAP.get(c['l4_id']),
            top3=[dict(l3=L3[j]['node_id'],label_ko=L3[j]['label_ko'],
                       sim=round(float(r['sim'][i,j]),4)) for j in t3[i]]))
    STAGE1[s]=out
    h=sum(1 for x in out if x['hold'])
    print(f'{s.upper()}: 자동 {len(out)-h} / hold {h} ({h/len(out)*100:.1f}%)')
json.dump(STAGE1,open(OUT/'stage1_assignment.json','w'),ensure_ascii=False,indent=1)
byc={s:{c['l4_id']:c for c in SETS[s]} for s in CFG['sets']}
for s in CFG['sets']:
    pay=[dict(l4_id=x['l4_id'],label_ko=byc[s][x['l4_id']]['label_ko'],label_en=byc[s][x['l4_id']]['label_en'],
              definition_ko=byc[s][x['l4_id']].get('definition_ko'),
              definition_en=byc[s][x['l4_id']].get('definition_en'),
              margin=x['margin'],top3=x['top3'],prev_l3=x['prev_l3'])
         for x in STAGE1[s] if x['hold']]
    json.dump(pay,open(DEC/('_input_holds_'+s+'.json'),'w'),ensure_ascii=False,indent=1)
    print('-> 🟢 Claude 입력',s,len(pay))
json.dump([dict(node_id=n['node_id'],label_ko=n['label_ko'],label_en=n['label_en'],
                definition_ko=n.get('definition_ko'),definition_en=n.get('definition_en'),
                parent_id=n['parent_id'],is_hold=n['node_id'] in HOLD_L3) for n in L3],
          open(DEC/'_input_l3_catalog.json','w'),ensure_ascii=False,indent=1)

MASTER: 자동 798 / hold 814 (50.5%)
C80: 자동 670 / hold 699 (51.1%)
C70: 자동 536 / hold 401 (42.8%)
-> 🟢 Claude 입력 master 814
-> 🟢 Claude 입력 c80 699
-> 🟢 Claude 입력 c70 401


In [8]:
# GATE-S2 — hold 심의 (🟢 Claude)
P=DEC/'hold_decisions.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 hold 심의 후 생성 ->',P)
    print('   형식 {"decisions":{"master":[{"l4_id":"...","l3":"RAI3-...","reason":"..."}],"c80":[...],"c70":[...]}}')
else:
    D=json.load(open(P))['decisions']; valid={n['node_id'] for n in L3}
    for s in CFG['sets']:
        ids={x['l4_id'] for x in STAGE1[s] if x['hold']}; got={d['l4_id'] for d in D.get(s,[])}
        assert not [d for d in D.get(s,[]) if d['l3'] not in valid], '미존재 L3'
        print(f'✅ {s}: hold {len(ids)} / 심의 {len(got)} / 미심의 {len(ids-got)}')

⏸  대기: 🟢 Claude가 hold 심의 후 생성 -> /Users/deep1003/data3/RAI-Risk-Taxonomy/data/experiments/stage2/decisions/hold_decisions.json
   형식 {"decisions":{"master":[{"l4_id":"...","l3":"RAI3-...","reason":"..."}],"c80":[...],"c70":[...]}}


In [13]:
# U5 — 2단계 강제 배정 (심의 고정 + 나머지 argmax, hold 없음)
HD=json.load(open(DEC/'hold_decisions.json'))['decisions']
FINAL={}
for s in CFG['sets']:
    man={d['l4_id']:d['l3'] for d in HD.get(s,[])}
    Xf=CARD[(s,BASE['card'])]; SD=SEEDS[BASE['seed']]
    fixed=np.array([c['l4_id'] in man for c in SETS[s]])
    fixedA=np.array([l3idx[man[c['l4_id']]] if c['l4_id'] in man else 0 for c in SETS[s]])
    r=dem(Xf,SD,BASE['alpha'],fixed=fixed,fixedA=fixedA)
    A=r['assign']; mg=r['margin']
    src=['adjudicated' if fixed[i] else ('auto' if not STAGE1[s][i]['hold'] else 'forced')
         for i in range(len(A))]
    FINAL[s]=[dict(l4_id=SETS[s][i]['l4_id'], l3=L3[A[i]]['node_id'], l3_label_ko=L3[A[i]]['label_ko'],
                   margin=round(float(mg[i]),4), source=src[i], prev_l3=OLDMAP.get(SETS[s][i]['l4_id']))
              for i in range(len(A))]
    print(f'{s.upper()}: {len(A)}장 | {dict(Counter(src))} | usedL3 {len(set(A))}/{len(L3)}'
          f' | max군집 {Counter(A.tolist()).most_common(1)[0][1]} | iters {r["iters"]}')
json.dump(FINAL,open(OUT/'stage2_assignment.json','w'),ensure_ascii=False,indent=1)

MASTER: 1612장 | {'adjudicated': 1612} | usedL3 54/56 | max군집 90 | iters 2
C80: 1369장 | {'adjudicated': 1369} | usedL3 54/56 | max군집 84 | iters 2
C70: 937장 | {'adjudicated': 937} | usedL3 54/56 | max군집 58 | iters 2


In [ ]:
# U6 — 2단계 민감도 3축 × 2회
rows2=[]; RES2={}
for s in CFG['sets']:
    man={d['l4_id']:d['l3'] for d in HD.get(s,[])}
    fixed=np.array([c['l4_id'] in man for c in SETS[s]])
    fixedA=np.array([l3idx[man[c['l4_id']]] if c['l4_id'] in man else 0 for c in SETS[s]])
    for ax,cv,sv,al in AXES:
        Xf=CARD[(s,cv)]; SD=SEEDS[sv]
        full=dem(Xf,SD,al,fixed=fixed,fixedA=fixedA); RES2[(s,ax)]=full
        mg=full['margin']
        rows2.append(dict(set=s,axis=ax,run='full',n=len(Xf),em_iters=full['iters'],
            median_margin=round(float(np.median(mg)),4),
            tiny_margin_pct=round(float((mg<0.01).mean()*100),1),
            used_l3=len(set(full['assign'])),empty_l3=len(L3)-len(set(full['assign'])),
            max_family=Counter(full['assign'].tolist()).most_common(1)[0][1],
            drift_min=round(float(full['drift'].min()),3)))
        for bs in CFG['boot_seeds']:
            rng=np.random.default_rng(bs)
            sel=np.sort(rng.choice(len(Xf),int(len(Xf)*CFG['boot_frac']),replace=False))
            rb=dem(Xf[sel],SD,al,fixed=fixed[sel],fixedA=fixedA[sel])
            rows2.append(dict(set=s,axis=ax,run=f'boot{bs}',n=len(sel),em_iters=rb['iters'],
                median_margin=round(float(np.median(rb['margin'])),4),
                tiny_margin_pct=round(float((rb['margin']<0.01).mean()*100),1),
                used_l3=len(set(rb['assign'])),empty_l3=len(L3)-len(set(rb['assign'])),
                max_family=Counter(rb['assign'].tolist()).most_common(1)[0][1],
                drift_min=round(float(rb['drift'].min()),3),
                boot_agreement_pct=round(float((rb['assign']==full['assign'][sel]).mean()*100),1)))
df2=pd.DataFrame(rows2); df2.to_csv(OUT/'stage2_sensitivity.csv',index=False)
print(df2.to_string(index=False))
st2=[]
for s in CFG['sets']:
    b=RES2[(s,'base')]['assign']
    for ax,_,_,_ in AXES:
        if ax=='base': continue
        a=RES2[(s,ax)]['assign']
        st2.append(dict(set=s,vs=ax,agreement_pct=round(float((a==b).mean()*100),1),
                        ARI=round(float(ari(b.tolist(),a.tolist())),3)))
df2s=pd.DataFrame(st2); df2s.to_csv(OUT/'stage2_stability.csv',index=False)
print(); print(df2s.to_string(index=False))

In [ ]:
# U7 — 검증 + 산출물 + HTML
import html as HH
valid={n['node_id'] for n in L3}; ok=True
for s in CFG['sets']:
    F=FINAL[s]
    assert len({x['l4_id'] for x in F})==len(SETS[s])
    assert all(x['l3'] in valid for x in F)
    hl=[x for x in F if x['l3'] in HOLD_L3]
    print(f'{s.upper()}: {len(F)}장 전원 배정 | HLD 잔류 {len(hl)}')
    if hl: ok=False
for s in CFG['sets']:
    with open(OUT/(s+'_l3_mapping.csv'),'w',newline='') as f:
        w=csv.writer(f); w.writerow(['l4_id','l3_id','l3_label_ko','margin','source','prev_l3'])
        for x in FINAL[s]: w.writerow([x['l4_id'],x['l3'],x['l3_label_ko'],x['margin'],x['source'],x['prev_l3']])
man=dict(run_at=datetime.datetime.now().isoformat(), cfg={k:v for k,v in CFG.items()},
    method='damped EM (alpha-anchored to enriched L3 seeds); random init excluded as non-identified',
    hierarchy_sha=sha(HIER), stage1_master_sha=sha(S1/'master.json'),
    versions=dict(python=platform.python_version(),numpy=np.__version__,scipy=scipy.__version__),
    counts={s:len(FINAL[s]) for s in CFG['sets']},
    l3_used={s:len({x['l3'] for x in FINAL[s]}) for s in CFG['sets']})
json.dump(man,open(OUT/'manifest.json','w'),ensure_ascii=False,indent=1,default=str)
cnts={s:Counter(x['l3'] for x in FINAL[s]) for s in CFG['sets']}
tr=''.join('<tr><td><code>'+n['node_id']+'</code></td><td>'+HH.escape(n['label_ko'])+'</td>'
           +''.join('<td class=num>'+str(cnts[t].get(n['node_id'],0))+'</td>' for t in CFG['sets'])+'</tr>'
           for n in L3)
CSS=("<style>body{font-family:'Apple SD Gothic Neo','Noto Sans KR',sans-serif;margin:18px}"
     "table{border-collapse:collapse;width:100%;font-size:12.5px}th,td{border:1px solid #ddd;padding:6px 8px}"
     "th{background:#f4f4f2}.num{text-align:right}code{color:#888}</style>")
open(OUT/'l3_distribution.html','w').write(
    '<!doctype html><meta charset=utf-8><title>Stage-2 L3 분포</title>'+CSS
    +'<h1>Stage-2 — L3 매핑 분포</h1><p>'
    +' · '.join(t.upper()+' '+str(len(FINAL[t]))+'장' for t in CFG['sets'])
    +' · 감쇠 EM α='+str(CFG['alpha'])+' · 보강 시드</p>'
    +'<table><tr><th>L3</th><th>명칭</th>'+''.join('<th>'+t.upper()+'</th>' for t in CFG['sets'])+'</tr>'+tr+'</table>')
print('\n완료 ->',OUT,'| 검증','PASS' if ok else 'FAIL')